# RAGU on Neo4j + Qdrant

Out of the box RAGU stores everything in the working directory: a `.gml` file for
the graph, JSON files for the vectors and the key-value data. That is fine for a
laptop and wrong for anything shared or large. `StorageArguments` is the single
place to change it — the pipeline, the engines and the `KnowledgeGraph` facade are
untouched.

| slot | backend | holds |
|---|---|---|
| graph | `Neo4jStorage` | entities and relations |
| vectors | `QdrantVectorDBStorage` | entity, relation and chunk vectors |
| key-value | `JsonKVStorage` | chunks, communities, summaries |


**Start the servers first:**

```
docker run -d -p 7687:7687 -e NEO4J_AUTH=neo4j/password neo4j:5
docker run -d -p 6333:6333 qdrant/qdrant
```

**Environment:** `OPENAI_API_KEY`, `LLM_MODEL_NAME`, `EMBEDDER_MODEL_NAME`, and
optionally `OPENAI_BASE_URL`, `NEO4J_URI`, `NEO4J_USER`, `NEO4J_PASSWORD`,
`QDRANT_URL` (all have localhost defaults).

In [ ]:
import os
from pathlib import Path

from ragu import (
    ArtifactsExtractorLLM,
    BuilderArguments,
    KnowledgeGraph,
    LocalSearchEngine,
    Settings,
    SimpleChunker,
    StorageArguments,
)
from ragu.models.embedder import EmbedderOpenAI
from ragu.models.llm import LLMOpenAI
from ragu.models.openai import CachedAsyncOpenAI
from ragu.search_engine.local_search import LocalParams
from ragu.storage.graph_storage_adapters.neo4j_adapter import Neo4jStorage
from ragu.storage.kv_storage_adapters.json_storage import JsonKVStorage
from ragu.storage.vdb_storage_adapters.qdrant_vdb import QdrantVectorDBStorage
from ragu.utils.ragu_utils import read_text_from_files

DATA_DIR = Path("data/en")
QUESTION = "Where did the father of the creator of the C programming language work?"

## Models

Qdrant needs the embedding dimension to create its collections, so resolve it
before building `StorageArguments`.

In [ ]:
Settings.language = "english"
# Still needed: the KV filenames and the Qdrant collection names derive from it.
Settings.storage_folder = "ragu_working_dir/neo4j_qdrant_example"

client = CachedAsyncOpenAI(
    base_url=os.environ.get("OPENAI_BASE_URL", "https://api.openai.com/v1"),
    api_key=os.environ["OPENAI_API_KEY"],
    rate_max_simultaneous=10,
    rate_max_per_minute=100,
)
llm = LLMOpenAI(client=client, model_name=os.environ["LLM_MODEL_NAME"])
embedder = EmbedderOpenAI(client=client, model_name=os.environ["EMBEDDER_MODEL_NAME"])
await embedder.initialize()

## Wire up the backends

`Index` adds `filename`, `node_cls` and `edge_cls` on top of `graph_storage_kwargs`.
`Neo4jStorage` ignores `filename`; it accepts the same argument bag as the
file-backed adapters so backends stay interchangeable.

**Do not put `collection_name` in `vdb_storage_kwargs`** — those kwargs are shared
by all three vector stores, so a fixed name would collapse entities, relations and
chunks into one collection. Left unset, each derives its own name from the filename
`Index` assigns it. `embedding_dim` is injected from the embedder.

In [ ]:
storage_settings = StorageArguments(
    graph_backend_storage=Neo4jStorage,
    graph_storage_kwargs={
        "uri": os.environ.get("NEO4J_URI", "bolt://localhost:7687"),
        "user": os.environ.get("NEO4J_USER", "neo4j"),
        "password": os.environ.get("NEO4J_PASSWORD", "testpassword"),
    },
    vdb_storage_type=QdrantVectorDBStorage,
    vdb_storage_kwargs={"url": os.environ.get("QDRANT_URL", "http://localhost:6333")},
    # Unchanged from the default, spelled out to show the third slot.
    kv_storage_type=JsonKVStorage,
)

knowledge_graph = KnowledgeGraph(
    llm=llm,
    embedder=embedder,
    chunker=SimpleChunker(max_chunk_size=1000),
    artifact_extractor=ArtifactsExtractorLLM(llm=llm, embedder=embedder),
    builder_settings=BuilderArguments(),
    storage_settings=storage_settings,
)

## Build

The expensive cell. Everything from here on reads what it wrote — re-run the query
cells without rebuilding.

In [ ]:
await knowledge_graph.build_from_docs(read_text_from_files(DATA_DIR))

## Audit

Cross-storage consistency: relations whose endpoints are missing, vectors with no
matching record, chunks referenced but never stored. Worth running against remote
backends, where a partial failure is easier to miss than with local files.

In [ ]:
print(await knowledge_graph.index.check_consistency())

## Query



In [ ]:
engine = LocalSearchEngine(llm=llm, knowledge_graph=knowledge_graph, embedder=embedder)
result = await engine.query(QUESTION, LocalParams(top_k=10))

print(f"Q: {QUESTION}")
print(f"A: {result.response}")

## Clean up

Neo4j and remote Qdrant both hold connection pools. A long-lived process that
builds indexes repeatedly leaks one per index without this.

In [ ]:
await knowledge_graph.index.close()